# Aethmodular Calibration Explorer: Colab launcher

## Goal

Mount Google Drive, unpack the prewarmed calibration-explorer bundle, validate the
required Drive data, start Flask, and open it through Colab's authenticated port proxy.

The prewarm bundle contains the AIRSpec and neutral-baseline caches, cohort tables,
shared scripts, filter dataset, and corrected-analog ranking cache. Disposable per-run CV-fit caches are omitted
to keep the download compact. The 4.6 GB FTIR source tree is **not duplicated**;
it is read from your mounted Drive.

> **Analog cutoff correction:** the bundled explorer counts TOR-eligible filters before
> applying cutoff N. The startup check should therefore report 500/500 for the locked raw
> analog cohort. Earlier cached 477/486 headline values must be regenerated under this rule.


## Setup

Run the next cells in order.

In [ ]:
from pathlib import Path

BUNDLE_NAME = "aethmodular_calibration_explorer_prewarm.zip"
DRIVE_FOLDER = Path("/content/drive/MyDrive/Aethmodular Colab")
BUNDLE_PATH = DRIVE_FOLDER / BUNDLE_NAME
WORK_ROOT = Path("/content/aethmodular")
PORT = 5058
print("Expected bundle:", BUNDLE_PATH)

### 1. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
if not BUNDLE_PATH.is_file():
    candidates = list(Path("/content/drive/MyDrive").rglob(BUNDLE_NAME))
    if len(candidates) == 1:
        BUNDLE_PATH = candidates[0]
    elif not candidates:
        raise FileNotFoundError(f"Upload {BUNDLE_NAME} to Google Drive first")
    else:
        raise RuntimeError(f"Multiple bundles found; set BUNDLE_PATH explicitly: {candidates}")
print("Using:", BUNDLE_PATH)

### 2. Unpack the prewarmed app

In [ ]:
import json
import shutil
import zipfile

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    archive.extractall(WORK_ROOT)
manifest = json.loads((WORK_ROOT / "PREWARM_MANIFEST.json").read_text())
print(json.dumps(manifest, indent=2))

### 3. Install the small runtime layer

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "Flask>=3,<4", "numpy>=2,<3", "pandas>=2,<3", "scipy>=1.13",
    "scikit-learn>=1.5", "matplotlib>=3.8", "pybaselines>=1.2",
    "polars>=1", "ikpls>=6.1",
])
print("Runtime installed")

### 4. Point the repo at mounted Drive and validate inputs

In [ ]:
import os
import sys

os.environ["AETHMODULAR_DRIVE_ROOT"] = "/content/drive/MyDrive"
sys.path.insert(0, str(WORK_ROOT))
# app.py uses bare `import hips_lab` / `import local_lab`, which resolve only
# when calibration_explorer/ itself is importable (locally it is the cwd)
sys.path.insert(0, str(WORK_ROOT / "calibration_explorer"))
sys.path.insert(0, str(WORK_ROOT / "research/ftir_hips_chem/scripts"))

from pls_transfer import FTIRTransferPaths

resolved = FTIRTransferPaths.defaults().validate()
display(resolved)
missing = resolved.loc[~resolved["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError(
        "Drive mounted, but required source paths were not found. "
        "Check AETHMODULAR_DRIVE_ROOT or your Drive folder layout:\n" + "\n".join(missing)
    )

## Start and open the explorer

In [ ]:
import importlib
import threading
import time
from werkzeug.serving import make_server

if "explorer_server" in globals():
    explorer_server.shutdown()

explorer = importlib.import_module("calibration_explorer.app")
for elapsed in range(240):
    if explorer.STATE["ready"] or explorer.STATE["error"]:
        break
    if elapsed % 10 == 0:
        print(explorer.STATE["message"])
    time.sleep(1)
if not explorer.STATE["ready"]:
    raise RuntimeError(explorer.STATE)

explorer_server = make_server("0.0.0.0", PORT, explorer.app, threaded=True)
explorer_thread = threading.Thread(target=explorer_server.serve_forever, daemon=True)
explorer_thread.start()
print("Explorer ready on port", PORT)
print("Startup checks:")
for check in explorer.STATE["checks"]:
    print("✓" if check["ok"] else "⚠", check["name"], "-", check["detail"])

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(PORT, height=900)

## Optional: spectral-comparison sweep over the whole library

Runs `research/ftir_ec_phase3/scripts/spectral_similarity.py` (the ftir_50 methods:
Hotelling T2 / Q residual, band-resolved correlation, spectral information divergence,
neighbourhood redundancy, selectivity-matched mutual k-NN) over the **entire**
13.6k-spectrum IMPROVE library against every SPARTAN target in the bundle. Pure
matmul/PCA: a couple of minutes on a Colab CPU, no GPU needed. Results land in
`research/ftir_ec_phase3/output/tables/ftir50_colab/`.

In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(WORK_ROOT / "research/ftir_ec_phase3/scripts"))
import spectral_similarity as ss

PHASE3 = WORK_ROOT / "research/ftir_ec_phase3"
OUT = PHASE3 / "output/tables/ftir50_colab"
OUT.mkdir(parents=True, exist_ok=True)

pool = np.load(PHASE3 / "output/corrected/improve_pool_corrected_df6.npz",
               allow_pickle=True)
LIB, WN = pool["corrected"].astype(float), pool["wn"].astype(float)
print(f"library: {LIB.shape[0]:,} spectra x {LIB.shape[1]} channels")

K = 50
rows = []
for target_dir in sorted((WORK_ROOT / "calibration_explorer/targets").iterdir()):
    corrected = target_dir / "spectra_corrected.csv"
    if not corrected.is_file():
        continue
    frame = pd.read_csv(corrected)
    cols = [c for c in frame.columns if c not in ("MediaId", "ExternalFilterId")]
    if not np.allclose(np.array([float(c) for c in cols]), WN):
        print(f"  skip {target_dir.name}: different wavenumber grid")
        continue
    Xt = frame[cols].to_numpy(float)
    dom = ss.pca_domain_diagnostics(LIB, Xt, n_components=10)
    red = ss.redundancy(Xt, LIB, k=K)
    mutual = ss.mutual_nearest(Xt, LIB, k=K)
    bands = ss.band_profile(Xt, np.median(LIB, axis=0)[None, :], WN)
    rows.append({
        "target": target_dir.name, "n": len(Xt),
        "T2_ratio_median": float(np.median(dom["T2_ratio"])),
        "Q_ratio_median": float(np.median(dom["Q_ratio"])),
        "pct_T2_over": 100 * float(np.mean(dom["T2_ratio"] > 1)),
        "pct_Q_over": 100 * float(np.mean(dom["Q_ratio"] > 1)),
        "unique_ratio": red["unique_ratio"],
        "pct_no_mutual": 100 * float(np.mean([len(m) == 0 for m in mutual])),
        **{f"r_{b}": float(np.median(v)) for b, v in bands.items()},
    })
    print(f"  {target_dir.name}: n={len(Xt)} done")

summary = pd.DataFrame(rows)
summary.to_csv(OUT / "spectral_comparison_all_targets.csv", index=False)
display(summary.round(3))

## Optional: exhaustive batch pre-compute

Runs the full cohort x cutoff-ladder x selection-space x spectra x protocol grid
**inside the server**, scoring every (configuration, k) row and appending it to
`calibration_explorer/cache/batch_results.jsonl`. Safe to leave running for hours;
progress prints below. Trim the GRID dict for a smaller pass.


In [ ]:
import time
import requests

# FLEET PATTERN: run several Colab sessions in parallel, each with a
# different slice of the grid (e.g. one protocol per session) -- the caches
# are content-keyed, so every session's cache zip merges cleanly back into
# calibration_explorer/cache/ on your own machine.
GRID = {
    "cohorts": ["eth_shaped", "analogs", "ocec", "smoke", "pool"],
    "spectra": ["raw", "airspec", "deriv2"],
    "modes": ["app"],         # THIS SESSION'S SLICE -- e.g. ["app"] here,
                              # ["app_fmm"] in a second session, etc.
    "corrsel": True,          # also select in AIRSpec-corrected space
    "cutoff_step": 10,        # dense mode: every 10th cutoff across the full
                              # range (eth 100-600, analogs 250-750,
                              # ocec 300-1500); set 0 for the 5-point ladder
    "cutoff_ladder": True,    # used only when cutoff_step is 0
    "sweep_k": True,          # sparse ladder always includes k=21 and k_max
    "k_min": 1,
    "k_max": 30,
    "dense_k": False,        # True = every integer 1..30 for every config (large)
    "lots": ["all", "251"],
    "match_eval_lot": True,  # train lot 251 -> Addis eval lot 251
    "target": "addis",
    "eval_lot": "all",
    "eval_group": "all",     # or one season, e.g. "Kiremt (Jun-Sep)"
    # ["early", "late"] scores every configuration on BOTH equal-n blind halves
    # in one pass (the fit is shared, so the second half costs a regression);
    # ["all"] keeps the whole evaluation set as before.
    "eval_splits": ["all"],
}
r = requests.post(f"http://127.0.0.1:{PORT}/api/batch_start", json=GRID, timeout=30).json()
print(r)
while True:
    s = requests.get(f"http://127.0.0.1:{PORT}/api/batch_status", timeout=30).json()
    print(f"{s['done']}/{s['total']}  {s['current']}  new rows: {s['new_rows']}"
          + (f"  failed: {s['skipped']}" if s["skipped"] else ""), flush=True)
    if not s["running"]:
        break
    time.sleep(30)
print("batch finished;", s["new_rows"], "rows saved")
if s.get("errors"):
    print("last errors:", *s["errors"], sep="\n  ")

### Copy the cache (and batch results) back to Drive

Everything the batch computed: CV-curve caches, per-k fits, and
`batch_results.jsonl`: zips back to your Drive folder. On your own machine,
unzip it into `calibration_explorer/cache/` (merging is safe: files are
content-keyed) and click **Load saved results** in the app's Optimize tab.


In [ ]:
import shutil
import time

stamp = time.strftime("%Y%m%d_%H%M")
archive = shutil.make_archive(
    str(DRIVE_FOLDER / f"explorer_cache_{stamp}"), "zip",
    root_dir=WORK_ROOT / "calibration_explorer", base_dir="cache")
print("cache copied to Drive:", archive)

## Checks and next steps

- Keep this Colab runtime open while using the explorer.
- Custom presets live in the browser's local storage, not in the bundle.
- New CV fits are cached in `/content/aethmodular/calibration_explorer/cache` for the
  current runtime. Copy that folder back to Drive if you want to preserve newly warmed
  configurations.
- The startup check for the locked raw analog cohort must report 500/500. A 477/500 result
  means an older bundle or server is still running.
